# Subword - Tokenization

Word tokenizers choke on unseen words. Character tokenizers blow up sequence length. Subword tokenizers split the difference.

## Problem Definition

Common words stay single tokens. Rare words decompose into meaningful pieces.

## Basic Concept

### BPE

Start with a character-level vocabulary. Count every adjacent pair, **merge the most frequent pair into a new token**

### Byte-level BPE

Same algorithm but over raw bytes (256 base tokens) instead of Unicode characters.

### Unigram

Start with a huge vocabulary, assgin each token a unigram probability, iteratively prune tokens whose removal least increases the corpus log-likelihood.

### WordPiece

Merge pairs that maximize likelihood of training corpus rather than raw frequency.

```
score(a, b) = freq(ab) / freq(a) / freq(b)
```

# SentencePiece vs tiktoken

SentencePiece is the library that trains vocabularies (BPE or Unigram) directly on raw Unicode text, encoding whitespace as _.

Tiktoken is OpenAI's fast encoder against pre-built vocabularies. It does not train.

# Build your Own

## BPE from scratch

In [5]:
import re
from collections import Counter
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

def word_counts(text):
    words = re.findall(r"[a-zA-Z]+", text.lower())
    return Counter(words)

def init_vocab(counts):
    return {tuple(word) + ("</w>",) : freq for word, freq in counts.items()}

def pair_counts(vocab):
    pairs = Counter()
    for symbols, freq in vocab.items():
        for a, b in zip(symbols, symbols[1:]):
            pairs[(a, b)] += freq
    return pairs

def merge_pair(vocab, pair):
    a, b = pair
    merged_symbol = a + b
    new_vocab = {}
    for symbols, freq in vocab.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_vocab[tuple(new_symbols)] = freq
    return new_vocab

def train_bpe(corpus, num_merges):
    counts = word_counts(corpus)
    if not counts:
        raise ValueError("word_counts: corpus produced no words")
    
    vocab = init_vocab(counts)
    merges = []
    for _ in range(num_merges):
        pairs = pair_counts(vocab)
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        merges.append(best)
        vocab = merge_pair(vocab, best)
    final_tokens = set()
    for symbols in vocab:
        final_tokens.update(symbols)
    return merges, sorted(final_tokens)

corpus = """
    the quick brown fox jumps over the lazy dog
    a stitch in time saves nine
    language models learn from statistical patterns in text
    tokenization splits text into smaller units called tokens
    subword tokenization lets rare words decompose into known pieces
    byte pair encoding is the dominant tokenization algorithm today
    the lazy dog slept while the fox jumped again and again
    patterns of letters in words are learnable and reusable
    """

with SectionPrinter("BPE from scratch"):
    merges, tokens = train_bpe(corpus, 50)
    print(merges)
    print(tokens)
            


======================BPE from scratch======================
[('e', '</w>'), ('s', '</w>'), ('n', '</w>'), ('t', 'i'), ('l', 'e'), ('t', 'o'), ('t', 'h'), ('th', 'e</w>'), ('i', 'n</w>'), ('n', 'i'), ('t', 'e'), ('d', '</w>'), ('a', 'n'), ('a', 'r'), ('a', 'ti'), ('a', 'l'), ('t', '</w>'), ('to', 'k'), ('tok', 'e'), ('i', 'n'), ('o', 'r'), ('m', 'p'), ('r', '</w>'), ('y', '</w>'), ('d', 'o'), ('g', '</w>'), ('a', 'g'), ('p', 'a'), ('t', 'te'), ('tte', 'r'), ('n', 's</w>'), ('toke', 'ni'), ('tokeni', 'z'), ('tokeniz', 'ati'), ('tokenizati', 'o'), ('tokenizatio', 'n</w>'), ('t', 's</w>'), ('w', 'or'), ('l', 'e</w>'), ('r', 'o'), ('w', 'n</w>'), ('f', 'o'), ('fo', 'x'), ('fox', '</w>'), ('j', 'u'), ('ju', 'mp'), ('v', 'e'), ('l', 'a'), ('la', 'z'), ('laz', 'y</w>')]
['</w>', 'a', 'ag', 'al', 'an', 'ar', 'ati', 'b', 'c', 'd', 'd</w>', 'do', 'e', 'e</w>', 'f', 'fox</w>', 'g', 'g</w>', 'h', 'i', 'in', 'in</w>', 'jump', 'k', 'l', 'lazy</w>', 'le', 'le</w>', 'm', 'mp', 'n', 'n</w>', 'ni', 'ns<

## Encode with learnedd merges

In [9]:
def encode_bpe(word, merges):
    symbols = list(word) + ["</w>"]
    for a, b in merges:
        i = 0
        while i < len(symbols) - 1:
            if symbols[i] == a and symbols[i + 1] == b:
                symbols = symbols[:i] + [a + b] + symbols[i + 2:]
            else:
                i += 1
    return symbols

with SectionPrinter("Encode with learned merges"):
    print(encode_bpe("quick", merges))
    print(encode_bpe("is", merges))
    print(encode_bpe("token", merges))

=================Encode with learned merges=================
['q', 'u', 'i', 'c', 'k', '</w>']
['i', 's</w>']
['toke', 'n</w>']
